# The quarterly deck that is rebuilt by hand every quarter

Someone exports the numbers, pastes them into last quarter's slides, updates the dates,
re-exports the charts, and notices on the third pass that one figure still has September's
band on it. It takes two days, it happens four times a year, and the most common defect is a
number and a chart that no longer come from the same fit.

A `Report` here is a `Spec` that holds **no data**. Every block that shows something names a
*context key* instead, and the data arrives at render time. That single choice is what
makes the thing a template rather than a document:

- **automated** — building a report is calling `render(template, context, format)`;
- **repeated** — the same template renders this quarter's numbers and next quarter's,
  and its content hash says the layout did not change between them;
- **styled** — a `Theme` is a value, so a house style is passed, not edited;
- **checkable** — `missing(template, context)` returns the names it still needs,
  before anything is drawn.

Three formats come out of one resolution pass, so they cannot disagree about what a
number is. **HTML** keeps plotly figures interactive and needs only plotly; **PPTX**
and **PDF** are static and need the `report` extra, returning a typed `Unsupported`
naming it when it is absent.

In [ ]:
import numpy as np
import pandas as pd

from axiom.core import LedgerLine, Unsupported, is_failure, summarize
from axiom.report import (
    FORMATS, GEOMETRY, AnyBlock, Divider, Figure, Format, Heading, LedgerBlock, Metric, PageBreak,
    PageSize, Paragraph, Report, ReportBuilder, ResolvedBlock, ResolvedFigure, ResolvedLedger,
    ResolvedMetric, ResolvedReport, ResolvedSection, ResolvedTable, ResolvedText, Section, Table,
    Theme, figure_png, missing, placeholders, render, render_html, render_pdf, render_pptx,
    resolve, write,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import HillKernel, fit, marginal_band, response_band

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

print("formats:", FORMATS, "| page sizes:", sorted(GEOMETRY))

## 1. Writing the template

`ReportBuilder` is the fluent surface: every method returns a new builder, nothing
mutates, and blocks attach to the section opened most recently. `{name}` and
`{name:.2f}` are placeholders; `placeholders` parses them, so a bad one fails when the
template is built rather than when it renders.

In [ ]:
template = (
    ReportBuilder("dose_readout", "Dose-response readout")
    .subtitle("Fitted {as_of}. Every number carries its interval.")
    .footer("Generated by axiom.report — template {template_hash}")
    .section("Headline", summary="What the fit says, and how sure it is.")
    .paragraph("The response **saturates**: the marginal effect at {top_dose:.0f} is "
               "already within rounding of zero.")
    .metric("contrast", "Expected outcome at 50", unit="units")
    .metric("sigma", "Residual sd", unit="units")
    .section("Curves", summary="Both figures carry a 90 % band, because they cannot not.")
    .figure("response", caption="Expected outcome against dose")
    .figure("marginal", caption="Marginal effect — where the band straddles zero, the model "
                                "does not know whether the next unit helps")
    .section("Detail")
    .table("arms", caption="Observed means by dose bucket", precision=2)
    .divider()
    .ledger("ledger", caption="What the numbers above rest on", show_detail=True)
    .build()
)
print("template hash :", template.content_hash()[:16])
print("sources       :", template.sources())
print("sections      :", [s.title for s in template.sections])
print("placeholders in the subtitle:", placeholders(template.subtitle))

The template contains no data at all — that is checkable, not just claimed.

In [ ]:
serialized = template.to_json()
print("template is", len(serialized), "bytes of layout")
print("contains a number from the data?", "7.2" in serialized)
print("round-trips:", Report.from_json(serialized) == template)
print("\nmissing from an empty context:")
table([[name] for name in missing(template, {})], headers=("unresolved key",))

## 2. The context

Anything the blocks can use: a float, an `Interval`, a `Summary`, an `EstimandResult`,
a `DataFrame`, a plotly figure, a `ResponseBand`, `LedgerLine`s. A `ResponseBand`
source is the interesting one — it is drawn through `viz.response_curve`, which cannot
draw a surface without its band.

In [ ]:
world = surface_world(
    n_units=5, n_periods=14, treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    doses=DosePlan(scale=50.0, spread=0.9, zero_fraction=0.2),
    intercept="shared", noise_sd=0.6, seed=7,
)
result = fit(world.spec, world.panel, backend="laplace", draws=600, chains=1, seed=7)
band = response_band(result, "a", n_grid=25, mass=0.9)
slope = marginal_band(result, "a", n_grid=25, mass=0.9)
doses = np.asarray(world.data["a"]).ravel()
frame = pd.DataFrame({"dose": doses, "outcome": np.asarray(world.panel.frame["y"])})
frame["bucket"] = pd.cut(frame["dose"], 4).astype(str)

context: dict[str, object] = {
    "as_of": "2026-08-21",
    "template_hash": template.content_hash()[:12],
    "top_dose": float(max(band.doses)),
    "contrast": summarize(np.asarray(result.posterior.draws("beta_a")).ravel(), mass=0.9),
    "sigma": float(result.posterior.summary("sigma").mean),
    "response": band,
    "marginal": slope,
    "arms": frame.groupby("bucket", as_index=False).agg(n=("outcome", "size"),
                                                        mean_outcome=("outcome", "mean")),
    "ledger": [
        LedgerLine(kind="identification", statement="dose is randomized; the adjustment set is empty",
                   detail={"route": "backdoor"}),
        LedgerLine(kind="interval",
                   statement=f"curves reported at {band.label()} over {band.n_draws} draws"),
    ],
}
print("missing now:", missing(template, context))

## 3. Resolution happens once

`resolve` fills the template from the context and produces renderer-neutral content, so
the three formats cannot differ on what a number is. It is also where the two
uncertainty rules live.

In [ ]:
resolved = resolve(template, context)
assert isinstance(resolved, ResolvedReport)
print(f"{'section':12s} {'blocks':>7}  kinds")
rows = []
for section in resolved.sections:
    assert isinstance(section, ResolvedSection)
    kinds = [type(b).__name__.removeprefix("Resolved") for b in section.blocks]
    rows.append([section.title, len(section.blocks), str(kinds)])
table(rows, headers=("section", "blocks", "kinds"))

metrics = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedMetric)]
figures = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedFigure)]
tables = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedTable)]
ledgers = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedLedger)]
texts = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedText)]
block: ResolvedBlock = metrics[0]
print("\nmetric with an interval  :", metrics[0].label, "=", metrics[0].text, metrics[0].interval_text)
print("metric without one       :", metrics[1].label, "=", metrics[1].text,
      "(no interval:", metrics[1].interval is None, ")")
print("figure traces            :", [len(f.figure.data) for f in figures],
      "-> band first:", [f.figure.data[0].fill == "toself" for f in figures])
print("table                    :", tables[0].columns, len(tables[0].rows), "rows")
print("ledger                   :", [line.kind for line in ledgers[0].lines])
print("placeholders filled      :", texts[0].text[:70], "...")

In [ ]:
figures[0].figure

In [ ]:
figures[1].figure

Those two figures are not illustrations of the report — they *are* the report's figure
blocks, resolved. The band came with them because `viz.response_curve` cannot draw a surface
without one, and the metric above them was summarized from the same posterior in the same
resolution pass. There is no step where a chart and a number can drift apart, because there
is no step.

## 4. Repeated: one template, two contexts, one hash

The point of a template. Change the data, keep the layout — and the content hash of the
template proves the layout is the same one.

In [ ]:
later = {**context, "as_of": "2026-11-30", "sigma": 0.71,
         "top_dose": float(max(band.doses)) * 1.1}
second = resolve(template, later)
assert isinstance(second, ResolvedReport)
print("same template hash:", resolved.source_hash == second.source_hash == template.content_hash())
print("different content :", resolved.sections[0].blocks[0].text != second.sections[0].blocks[0].text)
print("  first :", resolved.sections[0].blocks[0].text[-40:])
print("  second:", second.sections[0].blocks[0].text[-40:])

## 5. What it refuses

A gap in the *data* is a typed `Unsupported` naming every missing key at once. A
mistake in the *template* — a source of a type the block cannot use — is a `ValueError`
naming the block, because that is a bug rather than a gap.

In [ ]:
partial = render(template, {"as_of": "x"}, "html")
assert isinstance(partial, Unsupported)
print("missing keys:", partial.missing)
print("reason      :", partial.reason[:100], "...")

broken = Report(name="b", title="b",
                sections=(Section(title="s", blocks=(Figure(source="sigma"),)),))
try:
    resolve(broken, context)
except ValueError as e:
    print("\ntemplate mistake:", e)

## 6. Rendering

HTML is self-contained and **interactive**; PPTX and PDF are static and need the extra.
`write` infers the format from the suffix.

In [ ]:
import tempfile, pathlib
out = pathlib.Path(tempfile.mkdtemp())
rows = []
for fmt in FORMATS:
    chosen: Format = fmt
    written = write(template, context, str(out / f"readout.{fmt}"), chosen)
    if is_failure(written):
        rows.append([fmt, "refused", written.reason])
    else:
        rows.append([fmt, f"{pathlib.Path(written).stat().st_size:,} bytes", ""])
table(rows, headers=("format", "written", "or refused because"))

html = render_html(resolved)
assert isinstance(html, str)
print("\nHTML: interactive divs:", html.count('class="plotly-graph-div"'),
      "| band traces:", html.count('"fill":"toself"'),
      "| template hash embedded:", template.content_hash() in html)
print("HTML: the interval reached the page:", '(90% HDI)' in html or '(90% ETI)' in html)

## 7. Styling is a value

A `Theme` is a `Spec`, so restyling a report is a different report and the hash says
so. The same theme drives all three renderers — there is no stylesheet to drift.

In [ ]:
house = Theme(
    name="house", font="Helvetica", accent_color="#b5453b", muted_color="#6b7280",
    palette=("#b5453b", "#2f7fd1", "#3aa17e"), base_size=10.5, title_size=26.0,
    page="a4", slide="standard", figure_height=280.0,
)
size: PageSize = house.page
print("page:", size, house.page_geometry(), "| slide:", house.slide, house.slide_geometry())
print("heading sizes:", [house.heading_size(i) for i in (1, 2, 3)])
print("series colours:", [house.colour(i) for i in range(4)])
restyled = template.model_copy(update={"theme": house})
print("restyling changes the hash:", restyled.content_hash() != template.content_hash())
print("but not the sources      :", restyled.sources() == template.sources())

## 8. Building a template by hand

The builder is a convenience; the blocks are ordinary specs and a template can be
assembled, sliced or generated in a loop like any other data structure.

In [ ]:
blocks: tuple[AnyBlock, ...] = (
    Heading(text="Per-arm detail", level=2),
    Paragraph(text="One page per arm, generated in a loop.", emphasis=True),
    Metric(source="contrast", label="Effect", unit="units"),
    Table(source="arms"),
    Divider(),
    PageBreak(),
    LedgerBlock(source="ledger"),
)
generated = Report(
    name="generated", title="Generated", theme=house,
    sections=tuple(Section(title=f"Arm {i}", blocks=blocks) for i in range(1, 4)),
)
print("sections:", [s.title for s in generated.sections])
print("blocks per section:", [len(s.blocks) for s in generated.sections])
print("sources (deduplicated across sections):", generated.sources())
resolved_generated = resolve(generated, context)
assert isinstance(resolved_generated, ResolvedReport)
print("page breaks recorded at:", [sorted(s.breaks) for s in resolved_generated.sections])
deck = render_pptx(resolved_generated)
paged = render_pdf(resolved_generated)
table(
    [
        [label, f"{len(made):,} bytes" if isinstance(made, bytes) else made.reason]
        for label, made in (("pptx", deck), ("pdf", paged))
    ],
    headers=("format", "result"),
)
png = figure_png(figures[0].figure, house, height=220.0)
print("a figure as PNG:", f"{len(png):,} bytes" if isinstance(png, bytes) else png.reason)

## What this notebook decided

- A report is a `Spec` with **no data in it**. Blocks name context keys, so the same
  template runs on next quarter's numbers and its content hash proves the layout did
  not move.
- `missing` turns "will this render?" into a list of names, checked before anything is
  drawn. A missing key is `Unsupported`; a wrong *type* is a `ValueError` naming the
  block, because those are different mistakes.
- Resolution happens once for all three formats, so HTML, PPTX and PDF cannot disagree
  about what a number is.
- **Uncertainty is not a rendering option.** A `ResponseBand` source is drawn through
  `viz`, which cannot draw a surface without its band; a `Metric` whose source carries
  an interval prints it with its definition and mass. No field anywhere turns either
  off.
- HTML needs only plotly and stays interactive. PPTX and PDF need the `report` extra
  and say so by name when it is absent, rather than failing at import.

## What this bought you

A layout that is data, hashed, and reviewable; a rendering pass that fills it from a context
so the same template serves this quarter and next; three formats that cannot disagree because
they resolve once; and a refusal that names every missing key at once instead of rendering a
report with a hole in it.